# Implementing Softmax: A Coding Exercise

**Today's Challenge**: Implement the softmax function from scratch using PyTorch.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$$

In [37]:
import torch
import torch.nn.functional as F
import numpy as np

## Exercise 1: Implement Naive Softmax

**Task**: Implement softmax following the mathematical definition directly.

**Hint**: 
- Use `torch.exp()` for exponentials
- Use `.sum(dim=-1, keepdim=True)` to sum along the last dimension

In [38]:
def softmax_naive(x):
    """
    TODO: Implement softmax.
    Args:
        x: input tensor of shape (batch_size, num_classes) or (num_classes,)
    Returns:
        softmax probabilities
    """
    # YOUR CODE HERE
    pass

### Test Your Implementation

Run this cell to test with small values:

In [39]:
# Test case: Small values
x_small = torch.tensor([1.0, 2.0, 3.0])
print("Input:", x_small)
print("Your softmax:", softmax_naive(x_small))
print("PyTorch softmax:", F.softmax(x_small, dim=0))
print("Sum of probabilities:", softmax_naive(x_small).sum().item())

Input: tensor([1., 2., 3.])
Your softmax: None
PyTorch softmax: tensor([0.0900, 0.2447, 0.6652])


AttributeError: 'NoneType' object has no attribute 'sum'

---

## Solution 1: Naive Implementation

In [40]:
def softmax_naive(x):
    """
    Naive softmax implementation.
    """
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum(dim=-1, keepdim=True)

In [41]:
# Verify it works
x_small = torch.tensor([1.0, 2.0, 3.0])
print("Input:", x_small)
print("Naive softmax:", softmax_naive(x_small))
print("PyTorch softmax:", F.softmax(x_small, dim=0))
print("Match?", torch.allclose(softmax_naive(x_small), F.softmax(x_small, dim=0)))

Input: tensor([1., 2., 3.])
Naive softmax: tensor([0.0900, 0.2447, 0.6652])
PyTorch softmax: tensor([0.0900, 0.2447, 0.6652])
Match? True


---

## Exercise 2: Test with Large Values

**Task**: What happens when we use large input values? Run the cell below and observe.

In [42]:
# Test with large values
x_large = torch.tensor([1000.0, 1001.0, 1002.0])
print("Input:", x_large)
print("\nYour softmax:", softmax_naive(x_large))
print("PyTorch softmax:", F.softmax(x_large, dim=0))

Input: tensor([1000., 1001., 1002.])

Your softmax: tensor([nan, nan, nan])
PyTorch softmax: tensor([0.0900, 0.2447, 0.6652])


**Question**: What went wrong? Why do we get `nan` (not a number)?

*(Discuss with your neighbor for 2 minutes)*

---

## Solution 2: Understanding Overflow

In [43]:
# The problem: exp() overflows!
print("What happens when we compute exp(1000)?")
print(f"exp(1000) = {torch.exp(torch.tensor(1000.0))}")
print(f"\nThis is infinity! Maximum float32 value: {torch.finfo(torch.float32).max:.2e}")
print("\nWhen we compute inf / inf, we get nan")

What happens when we compute exp(1000)?
exp(1000) = inf

This is infinity! Maximum float32 value: 3.40e+38

When we compute inf / inf, we get nan


In [44]:
# When does overflow happen?
print("Testing different values:")
for val in [10, 50, 100, 500, 700, 710]:
    result = torch.exp(torch.tensor(float(val)))
    print(f"exp({val:3d}) = {result:.2e} {'<-- overflow!' if torch.isinf(result) else ''}")

Testing different values:
exp( 10) = 2.20e+04 
exp( 50) = 5.18e+21 
exp(100) = inf <-- overflow!
exp(500) = inf <-- overflow!
exp(700) = inf <-- overflow!
exp(710) = inf <-- overflow!


---

## Exercise 3: Test with Very Negative Values

**Task**: What about very negative values?

In [45]:
# Test with very negative values
x_negative = torch.tensor([-1000.0, -999.0, -998.0])
print("Input:", x_negative)
print("\nYour softmax:", softmax_naive(x_negative))
print("PyTorch softmax:", F.softmax(x_negative, dim=0))

Input: tensor([-1000.,  -999.,  -998.])

Your softmax: tensor([nan, nan, nan])
PyTorch softmax: tensor([0.0900, 0.2447, 0.6652])


**Question**: Why do we get `nan` here too?

*(Think for 30 seconds)*

---

## Solution 3: Understanding Underflow

In [46]:
# The problem: exp() underflows to 0!
print("What happens when we compute exp(-1000)?")
print(f"exp(-1000) = {torch.exp(torch.tensor(-1000.0))}")
print("\nAll exponentials become 0, so we compute 0 / 0 = nan")
print("\nExponentials of our values:")
x_negative = torch.tensor([-1000.0, -999.0, -998.0])
print(torch.exp(x_negative))

What happens when we compute exp(-1000)?
exp(-1000) = 0.0

All exponentials become 0, so we compute 0 / 0 = nan

Exponentials of our values:
tensor([0., 0., 0.])


---

## Exercise 4: Implement Stable Softmax

**Key Insight**: Softmax is **shift-invariant**:

$$\text{softmax}(x_i) = \text{softmax}(x_i + c) \text{ for any constant } c$$

**Proof**:
$$\frac{e^{x_i + c}}{\sum_j e^{x_j + c}} = \frac{e^c \cdot e^{x_i}}{e^c \cdot \sum_j e^{x_j}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

**Task**: Use this property to fix the numerical issues. What value should we subtract?

**Hint**: If we subtract the maximum value, the largest element becomes 0.

In [47]:
def softmax_stable(x):
    """
    TODO: Implement numerically stable softmax.
    Args:
        x: input tensor of shape (batch_size, num_classes) or (num_classes,)
    Returns:
        softmax probabilities
    """
    # YOUR CODE HERE
    pass

### Test Your Stable Implementation

In [48]:
# Test all three cases
test_cases = [
    ("Small values", torch.tensor([1.0, 2.0, 3.0])),
    ("Large values", torch.tensor([1000.0, 1001.0, 1002.0])),
    ("Negative values", torch.tensor([-1000.0, -999.0, -998.0]))
]

for name, x in test_cases:
    print(f"{name}:")
    print(f"  Your result: {softmax_stable(x)}")
    print(f"  PyTorch:     {F.softmax(x, dim=0)}")
    print(f"  Match? {torch.allclose(softmax_stable(x), F.softmax(x, dim=0))}")
    print()

Small values:
  Your result: None
  PyTorch:     tensor([0.0900, 0.2447, 0.6652])


TypeError: allclose(): argument 'input' (position 1) must be Tensor, not NoneType

---

## Solution 4: Stable Softmax

In [49]:
def softmax_stable(x):
    """
    Numerically stable softmax implementation.
    """
    # Subtract max for numerical stability
    x_max = x.max(dim=-1, keepdim=True)[0]
    x_shifted = x - x_max
    exp_x = torch.exp(x_shifted)
    return exp_x / exp_x.sum(dim=-1, keepdim=True)

### Why Does This Work?

In [50]:
x = torch.tensor([1000.0, 1001.0, 1002.0])
print("Original values:", x)
print("Max value:", x.max().item())
print()

x_shifted = x - x.max()
print("After subtracting max:", x_shifted)
print("Now the largest value is 0!")
print()

print("Exponentials of shifted values:")
print(torch.exp(x_shifted))
print()
print("These are safe:")
print(f"  exp(0) = {torch.exp(torch.tensor(0.0)):.6f}")
print(f"  exp(-1) = {torch.exp(torch.tensor(-1.0)):.6f}")
print(f"  exp(-2) = {torch.exp(torch.tensor(-2.0)):.6f}")
print("\nNo overflow (largest is exp(0) = 1)")
print("No underflow (denominators are non-zero)")

Original values: tensor([1000., 1001., 1002.])
Max value: 1002.0

After subtracting max: tensor([-2., -1.,  0.])
Now the largest value is 0!

Exponentials of shifted values:
tensor([0.1353, 0.3679, 1.0000])

These are safe:
  exp(0) = 1.000000
  exp(-1) = 0.367879
  exp(-2) = 0.135335

No overflow (largest is exp(0) = 1)
No underflow (denominators are non-zero)


### Verify All Test Cases

In [51]:
test_cases = [
    ("Small values", torch.tensor([1.0, 2.0, 3.0])),
    ("Large values", torch.tensor([1000.0, 1001.0, 1002.0])),
    ("Negative values", torch.tensor([-1000.0, -999.0, -998.0]))
]

print("Testing stable implementation:\n")
for name, x in test_cases:
    result = softmax_stable(x)
    pytorch = F.softmax(x, dim=0)
    print(f"{name}:")
    print(f"  Input:   {x}")
    print(f"  Output:  {result}")
    print(f"  Match?   {torch.allclose(result, pytorch)} ✓")
    print(f"  Sum:     {result.sum():.6f}")
    print()

Testing stable implementation:

Small values:
  Input:   tensor([1., 2., 3.])
  Output:  tensor([0.0900, 0.2447, 0.6652])
  Match?   True ✓
  Sum:     1.000000

Large values:
  Input:   tensor([1000., 1001., 1002.])
  Output:  tensor([0.0900, 0.2447, 0.6652])
  Match?   True ✓
  Sum:     1.000000

Negative values:
  Input:   tensor([-1000.,  -999.,  -998.])
  Output:  tensor([0.0900, 0.2447, 0.6652])
  Match?   True ✓
  Sum:     1.000000



---

## Bonus Exercise: Batched Inputs

**Task**: Verify your implementation works with batched inputs.

In [54]:
# Batched input: multiple samples at once
x_batch = torch.tensor([
    [1.0, 2.0, 3.0],           # sample 1: small values
    [1000.0, 1001.0, 1002.0],  # sample 2: large values
    [-1000.0, -999.0, -998.0]  # sample 3: negative values
])

print("Input shape:", x_batch.shape)
print("\nInput:")
print(x_batch)
print()

result = softmax_stable(x_batch)
print("Stable softmax output:")
print(result)
print()

print("Matches PyTorch?", torch.allclose(result, F.softmax(x_batch, dim=1)))
print("Row sums (should all be 1.0):", result.sum(dim=1))

Input shape: torch.Size([3, 3])

Input:
tensor([[ 1.0000e+00,  2.0000e+00,  3.0000e+00],
        [ 1.0000e+03,  1.0010e+03,  1.0020e+03],
        [-1.0000e+03, -9.9900e+02, -9.9800e+02]])

Stable softmax output:
tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])

Matches PyTorch? True
Row sums (should all be 1.0): tensor([1., 1., 1.])


---

## Advanced: Log-Softmax

Computing $\log(\text{softmax}(x))$ is common in classification (for NLL loss).

**Question**: Can we do better than computing softmax then taking log?

In [55]:
def log_softmax_stable(x):
    """
    Stable log-softmax:
    log(softmax(x_i)) = log(exp(x_i) / sum_j exp(x_j))
                      = x_i - log(sum_j exp(x_j))
                      = x_i - log(sum_j exp(x_j - max(x))) - max(x)
    """
    x_max = x.max(dim=-1, keepdim=True)[0]
    x_shifted = x - x_max
    log_sum_exp = torch.log(torch.exp(x_shifted).sum(dim=-1, keepdim=True))
    return x_shifted - log_sum_exp

In [28]:
# Test log-softmax
x = torch.tensor([1000.0, 1001.0, 1002.0])
print("Input:", x)
print("\nOur log-softmax:", log_softmax_stable(x))
print("PyTorch log_softmax:", F.log_softmax(x, dim=0))
print(f"\nMatch? {torch.allclose(log_softmax_stable(x), F.log_softmax(x, dim=0))}")

Input: tensor([1000., 1001., 1002.])

Our log-softmax: tensor([-2.4076, -1.4076, -0.4076])
PyTorch log_softmax: tensor([-2.4076, -1.4076, -0.4076])

Match? True


---

## Key Takeaways

1. **Naive softmax fails** with large or very negative values
   - Overflow: $e^{1000}$ = infinity
   - Underflow: $e^{-1000}$ = 0, leading to 0/0

2. **The fix**: Subtract max before computing exponentials
   - Uses softmax's shift invariance: $\text{softmax}(x) = \text{softmax}(x - c)$
   - Ensures largest value is 0, so largest exponential is 1
   - Prevents both overflow and underflow

3. **In practice**: Always use PyTorch's built-in functions
   - `F.softmax()` and `F.log_softmax()` are numerically stable
   - Optimized for performance
   - Support autograd, GPU, etc.

4. **Why this matters**: Understanding numerical stability is crucial for:
   - Implementing custom operations
   - Debugging training failures
   - Understanding why certain algorithms work better than others